# Stage 1: Read

In [4]:
import pandas as pd

df = pd.read_csv("orders_messy.csv", parse_dates=["order_date"])
df.dtypes

order_id               int64
city                  object
order_date    datetime64[ns]
revenue               object
cost                   int64
dtype: object

Anatomy check: `pd.read_csv(...)` is a function on the `pd` **module** (parens, it's an action), and `parse_dates` is the critical argument that turned `order_date` into a real `datetime64` instead of text.  Then, `df.dtypes` is an attribute--no parens, because you're just reading a property.

Notice what's already right and wrong: `order_date` is a proper datetime, but `revenue` came in as a `str` (object) because of the `$` and commas, and `cost` is correctly `int64`.  That tells us exactly what cleaning is needed.

# Stage 2: Inspect

In [6]:
df.shape

(9, 5)

In [7]:
df.isna().sum()

order_id      0
city          1
order_date    0
revenue       1
cost          0
dtype: int64

In [8]:
print(df["city"].value_counts())

city
Seattle     2
Portland    2
Boise       2
seattle     1
PORTLAND    1
Name: count, dtype: int64


In [9]:
df.duplicated().sum()

np.int64(0)

Anatomy check: `df.shape` is an attribute (no parens); `df.isna().sum()` chains two methods (both actions, both parens); `df["city"]` uses **brackets to pull one column into a Series**, then `.value_counts()` acts on it.

Second, look at the results carefully.  `value_counts()` exposes the casing mess--`Seattle` (2) and `seattle` (1) are counted *different* cities.  And notice `duplicated().sum()` returns $0$ even though order 1005 appears twice--because the two 1005 rows aren't *identical* (one has missing revenue, one has `$900`).  That's the lesson: "duplicate" depends on which columns you check.  We'll handle it with `subset=` in a moment.

# Stage 3: Clean

In [17]:
clean = df.copy()
clean["city"] = clean["city"].str.strip().str.title()
print(clean["city"].value_counts())

city
Seattle     3
Portland    3
Boise       2
Name: count, dtype: int64


In [19]:
clean.dtypes

order_id               int64
city                  object
order_date    datetime64[ns]
revenue               object
cost                   int64
dtype: object

In [20]:
clean["revenue"] = (
    clean["revenue"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype("float")
)
clean = clean.dropna(subset=["city"])
clean["revenue"] = clean["revenue"].fillna(0)
clean = clean.sort_values("revenue", ascending=False)
clean = clean.drop_duplicates(subset=["order_id"], keep="first")
clean = clean.sort_values("order_id").reset_index(drop=True)
clean["margin"] = clean["revenue"] - clean["cost"]

C:\Users\kellyja\AppData\Local\Temp\ipykernel_20608\1037016069.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean["revenue"] = clean["revenue"].fillna(0)


The `.copy()` first line (